In [5]:
# 0> 기본 라이브러리 세팅
import os 
import warnings
import pickle
from dotenv import load_dotenv

# 경고 메시지 삭제
warnings.filterwarnings('ignore')
load_dotenv()

# openai key 확인
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
	raise ValueError('.env 확인.. 키없음')

# 1> 필수 라이브러리 불러오기
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time

In [20]:
class SimpleRAGSystem:
    '''
    간단한 RAG 시스템 래퍼 클래스
    '''
    def __init__(self,vectorstore,llm, retriever_k=3):
        self.vectorstore=vectorstore
        self.llm = llm
        self.retriever=vectorstore.as_retriever(search_kwargs = {'k':retriever_k})
        self.chain = self._build_chain()
        
    def _build_chain(self):
        '''
        RAG체인 구성
        '''
        prompt = ChatPromptTemplate.from_messages([
            ('system','''당신은 제공된 문맥을 바탕으로 질문에 답변하는 AI입니다.
             문맥에 없는 정보는 답변하지 마세요.'''),
            ('human','문맥\n{content}\n\n질문:{question}\n\n답변:')
        ])
        return(
            {'content':self.retriever|self._format_docs,'question':RunnablePassthrough()}
            | prompt
            | self.llm
            | StrOutputParser()
        )
        
    @staticmethod
    # 데코레이터 : 클래스의 인스턴스(객체)에 속하지 않는 메서드를 정의할 때 사용
    # 데코레이터 사용하는이유 :
    #클래스 안에 포함시키고 싶은 함수지만, 인스턴스 상태와 무관할 때
    #유틸리티 함수나 클래스 관련 단순 계산 함수 정의할 때
    def _format_docs(docs):
        return '\n\n'.join([doc.page_conten for doc in docs])
    def ask(self,question:str) -> str:
        '''질문에 답변'''
        return self.chain.invoke(question)
    def ask_with_sources(self,question:str)->dict:
        '''질문에 답변 + 출처 반환'''
        answer = self.chain.invoke(question)
        sources = self.retriever.invoke(question)
        return {
        'answer':answer,
        'source':[ doc.metadata.get('source','unknown') for doc in sources]
        }
            


In [31]:
# %pip install chromadb
# %pip install --upgrade langchain

In [32]:
import langchain
import chromadb

print(langchain.__version__)
print(chromadb.__version__)


1.1.0
1.3.5


In [1]:
import sys
print(sys.executable)


c:\Users\Playdata2\miniconda3\envs\conda_venv\python.exe


In [22]:
from langchain_openai import OpenAIEmbeddings
import openai
from openai import OpenAI


if __name__ == '__main__':
    # vectorstore = # 적용할 VectorDB 입력 :
    #                 # 이전 단계 rag_1.2,rag_2.2,rag_3.2 에서
    #                 # 제작한 청킹rag_1.2 - 임베딩모델돌려서 저장한 벡터DB rag_2.2 , rag_3.2
    #                 # 불러와서 입력
    # llm = # 적용할 llm 모델 생성 혹은 불러오기
    embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-small')
    persist_dir = './chroma_db_reg2'
    vectorstore = Chroma(
        persist_directory = persist_dir,
        collection_name = 'persistent_rag',
        embedding_function = embedding_model
    )
    llm =  ChatOpenAI(
    model = 'gpt-4o-mini',
    temperature=0
    )

    rag_system = SimpleRAGSystem(vectorstore,llm)
    
    print("래퍼 클래스 테스트:")
    result = rag_system.ask_with_sources("VectorDB의 종류를 알려주세요")
    print(f"   질문: VectorDB의 종류를 알려주세요")
    print(f"   답변: {result['answer'][:100]}...")
    print(f"   출처: {result['source']}")
    
    # 다음에는 gpt에게 라이브러리 물어보지말자.. ;;

래퍼 클래스 테스트:
   질문: VectorDB의 종류를 알려주세요
   답변: 문맥에 VectorDB의 종류에 대한 정보가 포함되어 있지 않으므로, 해당 질문에 대한 답변을 제공할 수 없습니다....
   출처: []
